In [1]:
## Load the API token from .env file and configure the Genius client.

import os, time
import pandas as pd
from dotenv import load_dotenv
import lyricsgenius

# Load the Genius token
load_dotenv("../.env")
TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")
print("Token loaded:", TOKEN is not None)

# Create a Genius client
genius = lyricsgenius.Genius(
    TOKEN,
    timeout=15,  # seconds to wait for each request
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True,
)
genius.verbose = False

Token loaded: True


In [2]:
## Test the Genius API by searching for a song.

# Test
song = genius.search_song("Blinding Lights", "The Weeknd")
print(song.title, "|", song.url)
print(song.lyrics[:300])

Blinding Lights | https://genius.com/The-weeknd-blinding-lights-lyrics
Yeah

I've been tryna call
I've been on my own for long enough
Maybe you can show me how to love, maybe
I'm goin' through withdrawals
You don't even have to do too much
You can turn me on with just a touch, baby

I look around and
Sin City's cold and empty (Oh)
No one's around to judge me (Oh)
I can


In [3]:
# Load the Spotify track list from Part 1
spotify_df = pd.read_csv("../data/spotify_tracks.csv")
print(spotify_df.shape)

(94, 7)


In [4]:
## Clean song titles by removing featured artists and extra suffixes to improve search accuracy.

import re

def clean_title(t):
    # Improve search accuracy
    t = re.sub(r"\s*[\(\[](feat|with|ft)\.?[^\)\]]*[\)\]]", "", t, flags=re.I)
    t = re.sub(r"\s*-\s*.*$", "", t)
    return t.strip()

In [5]:
## Retrieve The Weeknd's song lyrics and data from Genius and store them in a DataFrame.

rows = []
for i, r in spotify_df.iterrows():
    query = clean_title(r["Song Name"])
    try:
        s = genius.search_song(query, "The Weeknd")
    except Exception as e:
        print("Error:", query, e)
        s = None

    # Only keep results where The Weeknd is the primary artist
    if s and "weeknd" in s.primary_artist["name"].lower():
        rows.append({
            "Album": s.album["name"] if s.album else None,  # some songs have no album on Genius
            "Song Name": s.title,
            "Lyrics": s.lyrics,
            # additional fields
            "Genius URL": s.url,  # link to song's page on Genius
            "Album Release Date": s.album.get("release_date_for_display") if s.album else None,
            "Annotation Count": s.annotation_count,  # number of user annotations
            "Spotify Song Name": r["Song Name"],  # original Spotify title used for search
        })
    else:
        print("Not found:", r["Song Name"])
    time.sleep(0.5)

genius_df = pd.DataFrame(rows)
print(genius_df.shape)
genius_df.head()

Not found: Until We're Skin & Bones
(93, 7)


,Album,Song Name,Lyrics,Genius URL,Album Release Date,Annotation Count,Spotify Song Name
0,Kiss Land,Professional,"It's ideal, oh\nYou need someone to tell you h...",https://genius.com/The-weeknd-professional-lyrics,"September 10, 2013",14,Professional
1,Kiss Land,The Town,"You did many things\nThat I liked, that I like...",https://genius.com/The-weeknd-the-town-lyrics,"September 10, 2013",8,The Town
2,Kiss Land,Adaptation,"When the sun comes up, you're searching for a ...",https://genius.com/The-weeknd-adaptation-lyrics,"September 10, 2013",12,Adaptation
3,Kiss Land,Love In The Sky,"There's no one inside, uh\nBut you're free to ...",https://genius.com/The-weeknd-love-in-the-sky-...,"September 10, 2013",20,Love In The Sky
4,Kiss Land,Belong To the World,The wind in the sky\nPlease gather the clouds\...,https://genius.com/The-weeknd-belong-to-the-wo...,"September 10, 2013",12,Belong To The World


In [6]:
# Find Spotify songs that were not found on Genius
set(spotify_df["Song Name"]) - set(genius_df["Spotify Song Name"])

{"Until We're Skin & Bones"}

In [7]:
# "Earned It" matched a remix page; replace it with the original version
s = genius.search_song("Earned It", "The Weeknd")
print(s.title, "|", s.url, "|", s.album["name"] if s.album else None)

Earned It | https://genius.com/The-weeknd-earned-it-lyrics | Beauty Behind the Madness


In [8]:
# Replace the remix row with the original song's data
idx = genius_df["Spotify Song Name"] == "Earned It (Fifty Shades Of Grey)"
genius_df.loc[idx, ["Album", "Song Name", "Lyrics", "Genius URL",
                    "Album Release Date", "Annotation Count"]] = [
    s.album["name"] if s.album else None, s.title, s.lyrics, s.url,
    s.album.get("release_date_for_display") if s.album else None,
    s.annotation_count]

# Confirm the row was updated
genius_df.loc[idx, ["Song Name", "Album", "Genius URL"]]

,Song Name,Album,Genius URL
18,Earned It,Beauty Behind the Madness,https://genius.com/The-weeknd-earned-it-lyrics


In [9]:
# Save the Genius data
genius_df.to_csv("../data/genius_lyrics.csv", index=False)
print("Saved to data/genius_lyrics.csv")

Saved to data/genius_lyrics.csv
